<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 一书（作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a>）的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 内存高效的模型权重加载（Memory-efficient Model Weight Loading）

- 本 notebook 提供一些技巧，用于在 GPU（或 CPU）内存有限时加载较大的 pretrained 或 finetuned models。
- 具体来说，它聚焦于这样一种场景：你用 `torch.save(model.state_dict(), "model.pth")` 保存模型（例如第 5-7 章中），之后想在新的 session 中加载它，用于继续 pretraining 或进一步 finetuning。
- 虽然示例使用的是 LLM，但本 notebook 解释的方法是通用的，适用于加载任何 PyTorch model，而不只限于 LLMs。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/memory-efficient-loading/memory-efficient-loading.webp" width="800px">

In [1]:
from importlib.metadata import version

pkgs = [
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

torch version: 2.9.1+cu130


&nbsp;
## 1. 基准测试工具（Benchmark utilities）

- 首先，定义一些 utility code 来跟踪 VRAM（GPU memory）。
- 稍后，我们还会引入一个工具来跟踪主系统 RAM（CPU memory）。
- 等后面实际应用这些函数时，它们的用途会更清楚。

In [2]:
import gc
import time
import torch


def start_memory_tracking():
    """Initialize GPU memory tracking."""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    else:
        print("This notebook is intended for CUDA GPUs but CUDA is not available.")

def print_memory_usage():
    max_gpu_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)  # Convert bytes to GB
    print(f"Maximum GPU memory allocated: {max_gpu_memory:.1f} GB")

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(3)  # some buffer time to allow memory to clear
    torch.cuda.reset_peak_memory_stats()
    max_memory_allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    print(f"Maximum GPU memory allocated: {max_memory_allocated:.1f} GB")

&nbsp;
## 2. 模型设置（Model setup）

- 这一代码部分会设置模型本身。
- 这里使用 "large" GPT-2 model，让示例更有意思（也可以使用 "gpt2-small (124M)" 来降低本 notebook 的内存需求和执行时间）。

In [3]:
from previous_chapters import GPTModel
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch04 import GPTModel



BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-xl (1558M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

- 现在看看 GPU memory functions 的实际效果：

In [4]:
start_memory_tracking()


model = GPTModel(BASE_CONFIG)
device = torch.device("cuda")
model.to(device)

print_memory_usage()

/home/rasbt/jupyterlab/reasoning/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


Maximum GPU memory allocated: 6.4 GB


- 另外，通过传入一个示例 tensor，确认模型可以正常运行：

In [5]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

- 接下来，假设我们正在 pretraining 模型，并把它保存下来供后续使用。
- 为简单起见，这里跳过实际 pretraining，只保存初始化后的模型（但概念是一样的）。

In [6]:
# Training code would go here...

model.train()
torch.save(model.state_dict(), "model.pth")

- 最后，在 Python session 中删除模型和示例 tensor，以重置 GPU memory。

In [7]:
del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 3. 基础权重加载（Basic weight loading）

- 现在进入有意思的部分：加载 pretrained model weights。
- 先看看加载之前保存的模型需要多少 GPU memory。

In [8]:
# Then load pretrained weights

start_memory_tracking()

model = GPTModel(BASE_CONFIG)
model.to(device)

model.load_state_dict(
    torch.load("model.pth", map_location=device, weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 12.8 GB


- 注意，这里的内存大约是上一 session 的 2 倍。
- 原因是短时间内同一个模型在内存中存在两份：
  - 第一份来自 `model.to(device)`。
  - 第二份来自代码行 `model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))`；最终，加载的 model weights 会复制进模型，`state_dict` 会被丢弃，但短时间内主模型和加载的 `state_dict` 会同时存在于内存中。
- 后续小节会重点解决这个问题。
- 但首先，先测试模型并重置 GPU memory。

In [9]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 再测试另一种实践中很常见的模式：

In [10]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG)

model.load_state_dict(
    torch.load("model.pth", map_location="cpu", weights_only=True)
)
model.to(device)
model.eval();

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB


In [11]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input
cleanup()

Maximum GPU memory allocated: 0.0 GB


- 因此，从 peak memory 来看，先在 device 上实例化模型再使用 `map_location="device"`，和先把权重加载到 CPU memory（`map_location="cpu"`）再移动到 device，并没有区别。

&nbsp;
## 4. 顺序加载权重（Loading weights sequentially）

- 上一节强调的问题是 model weights 会在 GPU memory 中短时间存在两份；一种 workaround 是顺序加载模型。
- 下面我们会：
  - 先把模型加载到 GPU memory
  - 再把 model weights 加载到 CPU memory
  - 最后逐个参数复制到 GPU memory

In [ ]:
start_memory_tracking()

model = GPTModel(BASE_CONFIG).to(device)

state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

print_memory_usage()

# Sequentially copy weights to the model's parameters
with torch.no_grad():
    for name, param in model.named_parameters():
        if name in state_dict:
            param.copy_(state_dict[name].to(device))
        else:
            print(f"Warning: {name} not found in state_dict.")

print_memory_usage()

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB


- 如上所示，内存使用量比之前低得多。
- 注意内存从 6.4 GB 增加到 6.7 GB，因为最初内存中只有模型，随后变成模型加 1 个 parameter tensor（我们临时把 parameter tensor 移到 GPU，以便用 `.to` 给模型赋值）。
- 总体来说，这是一个明显改进。
- 再次简短测试模型，然后为下一节重置 GPU memory。

In [ ]:
# Test if the model works (no need to track memory here)
test_input = torch.tensor([[1, 2, 3]]).to(device)
model.eval()

with torch.no_grad():
    model(test_input)

del model, test_input, state_dict, param
cleanup()

Maximum GPU memory allocated: 0.0 GB


&nbsp;
## 5. 用较低 CPU memory 加载模型（Loading the model with low CPU memory）

- 上一节中，我们通过先把权重（`state_dict`）加载到 CPU memory，再逐个复制到模型里，降低了 GPU memory 使用。
- 但是，如果 CPU memory 也有限怎么办？
- 本节使用 PyTorch 所谓的 `"meta"` device 方法，在 GPU memory 较大但 CPU memory 较小的机器上加载模型。
- 首先，定义一个 convenience function 来监控 CPU memory。

In [ ]:
import os
import psutil
from threading import Thread


def memory_usage_in_gb(func, *args, **kwargs):
    process = psutil.Process(os.getpid())

    # Measure the baseline memory usage before running the function
    baseline_mem = process.memory_info().rss / 1024 ** 3  # in GB

    # Start monitoring memory in a separate thread
    mem_usage = []
    done = False

    def monitor_memory():
        while not done:
            mem_usage.append(process.memory_info().rss / 1024 ** 3)  # Convert to GB
            time.sleep(0.1)

    t = Thread(target=monitor_memory)
    t.start()

    # Run the function
    func(*args, **kwargs)

    # Stop monitoring
    done = True
    t.join()

    peak_mem_usage_gb = max(mem_usage) - baseline_mem
    return peak_mem_usage_gb


- 首先，跟踪上一节顺序权重加载方法的 CPU memory 使用情况。

In [ ]:
def load_sequentially():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG).to(device)

    state_dict = torch.load("model.pth", map_location="cpu", weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name].to(device))
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_sequentially)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.7 GB
-> Maximum CPU memory allocated: 6.3 GB


- 现在假设有一台 CPU memory 较低但 GPU memory 较大的机器。
- 可以通过引入 PyTorch 所谓的 `"meta"` device，在 CPU memory 和 GPU memory 使用之间做取舍。
- PyTorch 的 meta device 是一种特殊 device type，允许创建不为实际数据分配内存的 tensors，本质上是创建 "meta" tensors。
- 这对模型分析或架构定义很有用：你需要 tensor shapes 和 types，但不想承担实际内存分配开销。

In [ ]:
def load_sequentially_with_meta():
    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    state_dict = torch.load("model.pth", map_location=device, weights_only=True)

    print_memory_usage()

    # Sequentially copy weights to the model's parameters
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in state_dict:
                param.copy_(state_dict[name])
            else:
                print(f"Warning: {name} not found in state_dict.")

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(load_sequentially_with_meta)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 1.3 GB


- 如上所示，通过在 meta-device 上创建模型，并直接把权重加载到 GPU memory，可以有效降低 CPU memory 需求。
- 你可能会问：“那顺序加载权重还有必要吗？它和原始方法相比如何？”
- 下面用本 notebook 第一个权重加载小节中的简单 PyTorch weight loading 方法进行对比。

In [ ]:
def baseline():
    start_memory_tracking()

    model = GPTModel(BASE_CONFIG)
    model.to(device)

    model.load_state_dict(torch.load("model.pth", map_location=device, weights_only=True))
    model.to(device)
    model.eval();

    print_memory_usage()

peak_memory_used = memory_usage_in_gb(baseline)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 12.8 GB
-> Maximum CPU memory allocated: 4.4 GB


- 如上所示，不使用 meta device 的“简单”权重加载会使用更多内存。
- 换句话说，如果机器 CPU memory 有限，可以使用 meta device 方法把 model weights 直接加载到 GPU memory，以降低 peak CPU memory usage。

&nbsp;
## 6. 使用 `mmap=True`（推荐）（Using `mmap=True`）

- 作为中级或高级 `torch.load` 用户，你可能想知道这些方法与 PyTorch 中的 `mmap=True` 设置相比如何。
- PyTorch 的 `mmap=True` 会启用 memory-mapped file I/O，让 tensor 直接从磁盘存储访问数据；如果 RAM 有限，这可以避免把整个文件加载进 RAM，从而减少内存使用。
- 也可以参考 [mikaylagawarecki](https://github.com/rasbt/LLMs-from-scratch/issues/402) 的有用评论。
- 乍看之下，它可能不如上面的顺序方法高效：

In [ ]:
def best_practices():
  with torch.device("meta"):
      model = GPTModel(BASE_CONFIG)

  model.load_state_dict(
      torch.load("model.pth", map_location=device, weights_only=True, mmap=True),
      assign=True
  )

  print_memory_usage()

peak_memory_used = memory_usage_in_gb(best_practices)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 5.9 GB


- CPU RAM 使用量如此之高的原因是：这台机器上有足够多的 CPU RAM 可用。
- 但是，如果在 CPU RAM 有限的机器上运行，`mmap` 方法会使用更少内存。

&nbsp;
## 7. 其他方法（Other methods）

- 本 notebook 聚焦于 PyTorch 中简单、内置的权重加载方法。
- 对于 CPU memory 有限的场景，推荐方法是上面解释的 `mmap=True` approach。
- 另外，还有一种 brute-force approach：分别保存和加载每个 weight tensor：

In [ ]:
model = GPTModel(BASE_CONFIG)
# Assume `model` is your trained model
state_dict = model.state_dict()

# Create a directory to store individual parameter files
os.makedirs("model_parameters", exist_ok=True)

# Save each parameter tensor separately
for name, param in state_dict.items():
    torch.save(param.cpu(), f"model_parameters/{name}.pt")

del model

In [ ]:
def load_individual_weights():

    start_memory_tracking()

    with torch.device("meta"):
        model = GPTModel(BASE_CONFIG)

    model = model.to_empty(device=device)

    print_memory_usage()
    param_dir = "model_parameters"

    with torch.no_grad():
        for name, param in model.named_parameters():
            weight_path = os.path.join(param_dir, f"{name}.pt")
            if os.path.exists(weight_path):
                param_data = torch.load(weight_path, map_location="cpu", weights_only=True)
                param.copy_(param_data)
                del param_data  # Free memory
            else:
                print(f"Warning: {name} not found in {param_dir}.")

    print_memory_usage()


peak_memory_used = memory_usage_in_gb(load_individual_weights)
print(f"-> Maximum CPU memory allocated: {peak_memory_used:.1f} GB")

Maximum GPU memory allocated: 6.4 GB
Maximum GPU memory allocated: 6.4 GB
-> Maximum CPU memory allocated: 0.3 GB
